# Solar Revolutions: paper-methods reproducibility notebook

This notebook implements the empirical methods described in the accepted paper section **"Revolutions and Solar Activity: An Empirical Analysis"**.

It is designed for Google Colab and for local reruns. It reconstructs the paper-style tests from public inputs:

- Revolutionary Episodes Dataset (RED) v1.0, 1900-2014.
- SILSO monthly sunspot numbers.
- World Bank world population.
- Peak-anchored solar-cycle phase.
- Rotational null tests that preserve historical clustering.
- Threshold-free weighted participation tests.
- Major-year threshold sensitivity.
- Alternative revolution measures.
- Leave-one-year, leave-one-cycle, and leave-one-cluster influence diagnostics.
- Optional monthly SILSO and Dst timing checks.

**Important limitation:** this notebook uses public RED v1.0. In our public reconstruction, RED v1.0 yields 343 episodes with start years in 1900-2014. If a manuscript version uses a different episode roster, exact numeric results may differ. The notebook is intended to make the method computable and auditable.


## 1. Hypotheses

### Primary empirical hypothesis

Large revolutionary mobilizations are not uniformly distributed through the solar cycle. They are unusually concentrated near the maximum phase of the cycle.

### Null hypothesis

The observed revolutionary sequence, including its historical clustering, is no closer to solar maxima than alternative circular alignments of the same sequence.

### Mechanism claim not directly tested here

This notebook does **not** prove a physiological, affective, geomagnetic, or causal mechanism. It tests whether the historical timing pattern exists under the paper's phase-alignment framework.


## 2. Colab setup

Run this cell first. It installs dependencies and downloads source files if they are not already present.


In [ ]:
#@title Install dependencies and fetch data { display-mode: "form" }
import os, json, math, csv, textwrap, warnings, subprocess, sys
from pathlib import Path
from urllib.request import Request, urlopen
from urllib.error import URLError, HTTPError

try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pandas', 'numpy', 'matplotlib', 'openpyxl', 'xlrd'])
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt

try:
    import openpyxl  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'openpyxl'])

warnings.filterwarnings('ignore')

ROOT = Path('/content/solar-revolutions-paper-methods') if 'google.colab' in sys.modules else Path.cwd() / 'solar-revolutions-colab-work'
DATA = ROOT / 'data'
OUT = ROOT / 'outputs'
DATA.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

RED_XLSM = DATA / 'Revolutionary episodes_v_1.0.xlsm'
RED_ZIP = DATA / 'revolutionary_episodes_v1.0.zip'
SUNSPOTS = DATA / 'sunspots_m.csv'
WORLD_POP = DATA / 'world_population_wb.json'
DST_MONTHLY = DATA / 'dst_monthly.csv'

DROPBOX_RED_URL = 'https://www.dropbox.com/scl/fi/vy6ibp9ggmjmmjye4brys/Revolutionary-episodes-dataset_v_1.0.zip?rlkey=2kp5ujvwq5augdtgaws10dc2y&dl=1'
SILSO_MONTHLY_URL = 'https://www.sidc.be/SILSO/DATA/SN_m_tot_V2.0.csv'
WORLD_POP_URL = 'https://api.worldbank.org/v2/country/WLD/indicator/SP.POP.TOTL?format=json&per_page=20000'

def download(url, dst):
    if dst.exists() and dst.stat().st_size > 0:
        return dst
    print('Downloading', dst.name)
    req = Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urlopen(req, timeout=120) as r:
        dst.write_bytes(r.read())
    return dst

# RED workbook
if not RED_XLSM.exists():
    download(DROPBOX_RED_URL, RED_ZIP)
    import zipfile
    with zipfile.ZipFile(RED_ZIP) as zf:
        names = zf.namelist()
        target = [n for n in names if n.endswith('.xlsm')][0]
        RED_XLSM.write_bytes(zf.read(target))

# SILSO monthly sunspot numbers
try:
    download(SILSO_MONTHLY_URL, SUNSPOTS)
except Exception as e:
    print('Could not download SILSO from SIDC:', repr(e))

# World population
if not WORLD_POP.exists():
    try:
        req = Request(WORLD_POP_URL, headers={'User-Agent': 'Mozilla/5.0'})
        obj = json.loads(urlopen(req, timeout=120).read().decode('utf-8'))
        pop = {int(row['date']): float(row['value']) for row in obj[1] if row.get('value') is not None}
        WORLD_POP.write_text(json.dumps(pop, indent=2))
    except Exception as e:
        print('Could not download World Bank population:', repr(e))

print('Working directory:', ROOT)
print('RED workbook:', RED_XLSM, RED_XLSM.exists())
print('SILSO monthly:', SUNSPOTS, SUNSPOTS.exists())
print('World population:', WORLD_POP, WORLD_POP.exists())


## 3. Load RED, SILSO, and population data

RED participant counts are assigned to the episode start year. Annual participation is divided by world population to match the paper's global-mobilization framing.


In [ ]:
#@title Load and construct annual revolutionary measures { display-mode: "form" }
START, END = 1900, 2014

def complete_population(raw, start=START, end=END):
    raw = {int(k): float(v) for k, v in raw.items() if v is not None}
    years = sorted(raw)
    out = {}
    for y in range(start, end + 1):
        if y in raw:
            out[y] = raw[y]
        elif y < years[0]:
            y0, y1 = years[0], years[1]
            g = raw[y1] / raw[y0]
            out[y] = raw[y0] / (g ** (y0 - y))
        elif y > years[-1]:
            y0, y1 = years[-2], years[-1]
            g = raw[y1] / raw[y0]
            out[y] = raw[y1] * (g ** (y - y1))
        else:
            lo = max(k for k in years if k < y)
            hi = min(k for k in years if k > y)
            frac = (y - lo) / (hi - lo)
            out[y] = raw[lo] + frac * (raw[hi] - raw[lo])
    return out

pop = complete_population(json.loads(WORLD_POP.read_text()))

timing = pd.read_excel(RED_XLSM, sheet_name='1-Timing & location')
participants = pd.read_excel(RED_XLSM, sheet_name='5-Participants')
participant_by_id = dict(zip(participants['revid'].astype(str), participants['particnum'].fillna(0).astype(float)))

annual = {y: {'year': y, 'events': 0, 'participants': 0.0, 'largest_episode': 0.0, 'episode_names': []} for y in range(START, END + 1)}
for _, row in timing.iterrows():
    if pd.isna(row.get('startyear')):
        continue
    y = int(row['startyear'])
    if not (START <= y <= END):
        continue
    rid = str(int(row['revid'])) if not pd.isna(row.get('revid')) else str(row.get('revid'))
    p = float(participant_by_id.get(rid, 0.0) or 0.0)
    annual[y]['events'] += 1
    annual[y]['participants'] += p
    annual[y]['largest_episode'] = max(annual[y]['largest_episode'], p)
    annual[y]['episode_names'].append(str(row.get('nameofrevolution', '')))

df = pd.DataFrame([annual[y] for y in range(START, END + 1)])
df['world_population'] = df['year'].map(pop)
df['participation_share'] = df['participants'] / df['world_population']
df['largest_episode_share'] = df['largest_episode'] / df['world_population']
df['event_count'] = df['events']
df['log1p_participants'] = np.log1p(df['participants'])
# Composite index: standardize participation share and event counts, then average. This is transparent and parameter-light.
df['composite_index'] = ((df['participation_share'] - df['participation_share'].mean()) / df['participation_share'].std(ddof=0) +
                         (df['event_count'] - df['event_count'].mean()) / df['event_count'].std(ddof=0)) / 2

print('RED episodes with start years 1900-2014:', int(df['events'].sum()))
print('Years:', df['year'].min(), df['year'].max(), 'n=', len(df))
df[['year','events','participants','participation_share','largest_episode']].sort_values('participants', ascending=False).head(12)


## 4. Solar-cycle phase

The paper aligns each solar cycle by phase: minimum = 0, maximum = π, next minimum = 2π. This makes every recognized maximum the same phase even when cycles differ in length.

The cycle extrema below are encoded explicitly so the notebook is stable and auditable. They can be replaced with a table derived directly from smoothed SILSO monthly values if desired.


In [ ]:
#@title Define peak-anchored solar phase { display-mode: "form" }
CYCLES = [
    {'cycle': 13, 'min0': 1890.2, 'max': 1893.9, 'min1': 1902.0},
    {'cycle': 14, 'min0': 1902.0, 'max': 1906.2, 'min1': 1913.6},
    {'cycle': 15, 'min0': 1913.6, 'max': 1917.6, 'min1': 1923.6},
    {'cycle': 16, 'min0': 1923.6, 'max': 1928.3, 'min1': 1933.8},
    {'cycle': 17, 'min0': 1933.8, 'max': 1937.4, 'min1': 1944.2},
    {'cycle': 18, 'min0': 1944.2, 'max': 1947.5, 'min1': 1954.3},
    {'cycle': 19, 'min0': 1954.3, 'max': 1958.2, 'min1': 1964.8},
    {'cycle': 20, 'min0': 1964.8, 'max': 1968.9, 'min1': 1976.5},
    {'cycle': 21, 'min0': 1976.5, 'max': 1979.9, 'min1': 1986.8},
    {'cycle': 22, 'min0': 1986.8, 'max': 1989.6, 'min1': 1996.4},
    {'cycle': 23, 'min0': 1996.4, 'max': 2001.8, 'min1': 2008.9},
    {'cycle': 24, 'min0': 2008.9, 'max': 2014.3, 'min1': 2019.9},
]

def phase_for_decimal_year(t):
    for c in CYCLES:
        if c['min0'] <= t < c['min1']:
            if t <= c['max']:
                theta = math.pi * ((t - c['min0']) / (c['max'] - c['min0']))
            else:
                theta = math.pi + math.pi * ((t - c['max']) / (c['min1'] - c['max']))
            return theta, c['cycle'], abs(t - c['max']), c['max']
    raise ValueError(f'Year outside cycle table: {t}')

def phase_for_year(y):
    return phase_for_decimal_year(y + 0.5)

phase_rows = []
for y in df['year']:
    theta, cycle, distance, max_year = phase_for_year(int(y))
    phase_rows.append((theta, cycle, distance, max_year, 100 * theta / (2 * math.pi)))

df[['theta','solar_cycle','distance_to_max_years','cycle_max_year','cycle_phase_pct']] = pd.DataFrame(phase_rows, index=df.index)

df[['year','solar_cycle','cycle_phase_pct','distance_to_max_years']].head()


## 5. Primary threshold-free phase test

For annual revolutionary weights \(w_t\), the directional statistic is:

\[T_w = \frac{\sum_t w_t \cos(\theta_t - \pi)}{\sum_t w_t}\]

It approaches 1 when revolutionary participation is concentrated at solar maxima, 0 under no preferred phase, and -1 near minima.

The rotational null holds the solar cycle fixed and circularly shifts the revolutionary sequence through every alternative annual alignment. This preserves the historical clustering of revolutions.


In [ ]:
#@title Primary statistic and rotational null { display-mode: "form" }
def phase_stats(weights, theta, distance):
    weights = np.asarray(weights, dtype=float)
    theta = np.asarray(theta, dtype=float)
    distance = np.asarray(distance, dtype=float)
    total = weights.sum()
    if total <= 0:
        return {'T_w': np.nan, 'weighted_distance_years': np.nan}
    return {
        'T_w': float((weights * np.cos(theta - math.pi)).sum() / total),
        'weighted_distance_years': float((weights * distance).sum() / total),
    }

def rotational_test(weights, theta, distance):
    weights = np.asarray(weights, dtype=float)
    obs = phase_stats(weights, theta, distance)
    null_T, null_D = [], []
    for shift in range(1, len(weights)):
        wr = np.roll(weights, shift)
        s = phase_stats(wr, theta, distance)
        null_T.append(s['T_w'])
        null_D.append(s['weighted_distance_years'])
    null_T = np.array(null_T)
    null_D = np.array(null_D)
    return {
        **obs,
        'p_T_rotational': float((null_T >= obs['T_w']).mean()),
        'p_distance_rotational': float((null_D <= obs['weighted_distance_years']).mean()),
        'n_rotations': int(len(null_T)),
        'null_T': null_T,
        'null_D': null_D,
    }

primary = rotational_test(df['participation_share'], df['theta'], df['distance_to_max_years'])
print('Primary participation-share test')
for k in ['T_w','weighted_distance_years','p_T_rotational','p_distance_rotational','n_rotations']:
    print(f'{k}: {primary[k]}')

pd.DataFrame({
    'metric': ['T_w', 'weighted distance to max', 'rotational p for T_w', 'rotational p for distance'],
    'value': [primary['T_w'], primary['weighted_distance_years'], primary['p_T_rotational'], primary['p_distance_rotational']],
})


## 6. Visualize annual participation and solar phase

These figures are descriptive. The inferential test is the rotational null above.


In [ ]:
#@title Figures: annual participation and phase histogram { display-mode: "form" }
# Annual SILSO for plotting.
def load_silso_monthly(path):
    rows = []
    for line in path.read_text(errors='ignore').splitlines():
        if not line.strip() or line.startswith('#'):
            continue
        parts = [p.strip() for p in line.split(';')]
        if len(parts) >= 4:
            y, m, dec, val = int(parts[0]), int(parts[1]), float(parts[2]), float(parts[3])
            if val >= 0:
                rows.append({'year': y, 'month': m, 'decimal_year': dec, 'sunspot': val})
    return pd.DataFrame(rows)

sun_m = load_silso_monthly(SUNSPOTS)
sun_y = sun_m[(sun_m['year'] >= START) & (sun_m['year'] <= END)].groupby('year', as_index=False)['sunspot'].mean()
plot_df = df.merge(sun_y, on='year', how='left')

fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.plot(plot_df['year'], plot_df['sunspot'], color='0.55', lw=1.5, label='SILSO annual mean')
ax1.set_ylabel('Sunspot number')
ax2 = ax1.twinx()
ax2.bar(plot_df['year'], plot_df['participation_share'] * 100, color='#b14d3a', alpha=0.55, label='RED participation share')
ax2.set_ylabel('Revolutionary participation (% world population)')
for c in CYCLES:
    if START <= c['max'] <= END:
        ax1.axvline(c['max'], color='black', alpha=0.18, lw=1)
ax1.set_title('RED participation and SILSO solar cycle, 1900-2014')
ax1.set_xlabel('Year')
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 4))
phase_bins = np.linspace(0, 100, 13)
weighted, edges = np.histogram(df['cycle_phase_pct'], bins=phase_bins, weights=df['participation_share'])
ax.bar(edges[:-1], weighted / weighted.sum(), width=np.diff(edges), align='edge', color='#b14d3a', edgecolor='white')
ax.axvline(50, color='black', ls='--', label='Solar maximum')
ax.set_xlabel('Solar cycle phase (%)')
ax.set_ylabel('Share of total revolutionary participation weight')
ax.set_title('Participation weight by peak-anchored solar-cycle phase')
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(primary['null_T'], bins=25, color='0.8', edgecolor='white')
ax.axvline(primary['T_w'], color='#b14d3a', lw=3, label=f"observed T_w={primary['T_w']:.3f}")
ax.set_title('Rotational null distribution for T_w')
ax.set_xlabel('T_w under circular shifts')
ax.set_ylabel('Count')
ax.legend()
plt.show()


## 7. Alternative annual measures

This reproduces the paper's key robustness idea: the result should be checked across several ways of measuring revolutionary activity.


In [ ]:
#@title Alternative revolutionary measures { display-mode: "form" }
measures = {
    'Participation share of world population': 'participation_share',
    'Largest single episode share': 'largest_episode_share',
    'Composite index shifted positive': 'composite_index',
    'Episode count alone': 'event_count',
    'log(1 + participants)': 'log1p_participants',
}

rows = []
for label, col in measures.items():
    w = df[col].copy().astype(float)
    # The T_w statistic requires nonnegative weights. Shift the standardized composite if needed.
    if w.min() < 0:
        w = w - w.min()
    s = rotational_test(w, df['theta'], df['distance_to_max_years'])
    rows.append({
        'measure': label,
        'weighted_distance_years': s['weighted_distance_years'],
        'p_distance_rotational': s['p_distance_rotational'],
        'T_w': s['T_w'],
        'p_T_rotational': s['p_T_rotational'],
    })

measure_results = pd.DataFrame(rows)
measure_results.to_csv(OUT / 'paper_methods_alternative_measures.csv', index=False)
measure_results


## 8. Major-year threshold sensitivity

The paper treats major-year thresholds as illustrative, not primary. This section checks whether high-participation years cluster near maximum under several threshold rules.


In [ ]:
#@title Major-year threshold sensitivity { display-mode: "form" }
def unweighted_phase_test(mask):
    sub = df[mask].copy()
    w = np.ones(len(sub))
    return sub, rotational_test(w, sub['theta'], sub['distance_to_max_years'])

x = df['participation_share']
threshold_specs = {
    'mean + 1 SD': x > (x.mean() + x.std(ddof=0)),
    'top 15%': x >= x.quantile(0.85),
    'top 10%': x >= x.quantile(0.90),
    'top 5%': x >= x.quantile(0.95),
}

threshold_rows = []
major_year_lists = {}
for name, mask in threshold_specs.items():
    sub, s = unweighted_phase_test(mask)
    major_year_lists[name] = list(sub['year'].astype(int))
    threshold_rows.append({
        'definition': name,
        'n_years': len(sub),
        'years': ', '.join(map(str, major_year_lists[name])),
        'mean_distance_years': s['weighted_distance_years'],
        'p_distance_rotational': s['p_distance_rotational'],
        'T': s['T_w'],
        'p_T_rotational': s['p_T_rotational'],
    })

threshold_results = pd.DataFrame(threshold_rows)
threshold_results.to_csv(OUT / 'paper_methods_threshold_sensitivity.csv', index=False)
threshold_results


## 9. Influence diagnostics

This section answers whether the result depends on a single year, solar cycle, or historical wave.


In [ ]:
#@title Leave-one-year, leave-one-cycle, and leave-one-cluster diagnostics { display-mode: "form" }
baseline = primary

def summarize_removal(remove_years):
    w = df['participation_share'].copy().astype(float).to_numpy()
    remove_years = set(remove_years)
    removed = df['year'].isin(remove_years).to_numpy()
    removed_weight = float(w[removed].sum())
    w[removed] = 0.0
    s = rotational_test(w, df['theta'], df['distance_to_max_years'])
    return s, removed_weight

leave_year = []
for y in df['year'].astype(int):
    s, removed_weight = summarize_removal([y])
    row = df.loc[df['year'] == y].iloc[0]
    leave_year.append({
        'removed_year': y,
        'participants': int(row['participants']),
        'events': int(row['events']),
        'solar_cycle': int(row['solar_cycle']),
        'weight_share_pct': 100 * removed_weight / float(df['participation_share'].sum()),
        'leave_one_T_w': s['T_w'],
        'delta_T_w': s['T_w'] - baseline['T_w'],
        'leave_one_distance_years': s['weighted_distance_years'],
        'delta_distance_years': s['weighted_distance_years'] - baseline['weighted_distance_years'],
        'leave_one_p_T': s['p_T_rotational'],
    })
leave_year = pd.DataFrame(leave_year).sort_values('delta_T_w', key=lambda s: s.abs(), ascending=False)
leave_year.to_csv(OUT / 'paper_methods_leave_one_year.csv', index=False)

leave_cycle = []
for cycle in sorted(df['solar_cycle'].unique()):
    years = list(df.loc[df['solar_cycle'] == cycle, 'year'].astype(int))
    s, removed_weight = summarize_removal(years)
    leave_cycle.append({
        'removed_cycle': int(cycle),
        'years_removed': f'{min(years)}-{max(years)}',
        'weight_share_pct': 100 * removed_weight / float(df['participation_share'].sum()),
        'leave_one_T_w': s['T_w'],
        'delta_T_w': s['T_w'] - baseline['T_w'],
        'leave_one_distance_years': s['weighted_distance_years'],
        'delta_distance_years': s['weighted_distance_years'] - baseline['weighted_distance_years'],
        'leave_one_p_T': s['p_T_rotational'],
    })
leave_cycle = pd.DataFrame(leave_cycle)
leave_cycle.to_csv(OUT / 'paper_methods_leave_one_cycle.csv', index=False)

clusters = {
    '1905 Russian Revolution wave': [1905],
    '1917-1920 WWI/postwar revolutionary wave': list(range(1917, 1921)),
    '1946 postwar/decolonization wave': [1946],
    '1968 global protest wave': [1968],
    '1978-1980 Iran/Nicaragua/Latin America wave': list(range(1978, 1981)),
    '1988-1991 late Cold War wave': list(range(1988, 1992)),
    '2011-2013 Arab Spring/late RED wave': list(range(2011, 2014)),
    '1917-1920 and 1988-1991 combined': list(range(1917, 1921)) + list(range(1988, 1992)),
}
leave_cluster = []
for name, years in clusters.items():
    s, removed_weight = summarize_removal(years)
    leave_cluster.append({
        'removed_cluster': name,
        'years_removed': ', '.join(map(str, years)),
        'weight_share_pct': 100 * removed_weight / float(df['participation_share'].sum()),
        'leave_one_T_w': s['T_w'],
        'delta_T_w': s['T_w'] - baseline['T_w'],
        'leave_one_distance_years': s['weighted_distance_years'],
        'delta_distance_years': s['weighted_distance_years'] - baseline['weighted_distance_years'],
        'leave_one_p_T': s['p_T_rotational'],
    })
leave_cluster = pd.DataFrame(leave_cluster)
leave_cluster.to_csv(OUT / 'paper_methods_leave_one_cluster.csv', index=False)

print('Top leave-one-year effects')
display(leave_year.head(12))
print('Leave-one-cluster effects')
display(leave_cluster)


## 10. Monthly timing check

This is not the paper's main test. It asks a stricter question: do RED episode start months rise contemporaneously with monthly SILSO? In prior runs this was weak/null, which suggests the paper's result is a broad cycle-phase pattern rather than a month-to-month trigger.


In [ ]:
#@title Monthly RED starts vs monthly SILSO { display-mode: "form" }
monthly_timing = timing.copy()
monthly_timing = monthly_timing[(monthly_timing['startyear'] >= START) & (monthly_timing['startyear'] <= END)].copy()
monthly_timing['startdate_parsed'] = pd.to_datetime(monthly_timing['startdate'], errors='coerce')
monthly_timing['month'] = monthly_timing['startdate_parsed'].dt.month.fillna(7).astype(int)
monthly_counts = monthly_timing.groupby(['startyear','month']).size().reset_index(name='red_starts')
monthly_counts['date'] = pd.to_datetime(dict(year=monthly_counts['startyear'], month=monthly_counts['month'], day=1))

month_index = pd.date_range(f'{START}-01-01', f'{END}-12-01', freq='MS')
monthly_df = pd.DataFrame({'date': month_index})
monthly_df['year'] = monthly_df['date'].dt.year
monthly_df['month'] = monthly_df['date'].dt.month
monthly_df = monthly_df.merge(monthly_counts[['date','red_starts']], on='date', how='left')
monthly_df['red_starts'] = monthly_df['red_starts'].fillna(0)

silso_m = load_silso_monthly(SUNSPOTS)
silso_m['date'] = pd.to_datetime(dict(year=silso_m['year'], month=silso_m['month'], day=1))
monthly_df = monthly_df.merge(silso_m[['date','sunspot']], on='date', how='left')

monthly_rows = []
for window in [0, 1, 3, 6, 12, 24]:
    if window == 0:
        x = monthly_df['sunspot']
        label = 'same month'
    else:
        x = monthly_df['sunspot'].rolling(window + 1, min_periods=1).mean()
        label = f'current + prior {window} months'
    y = monthly_df['red_starts']
    pearson = float(pd.Series(x).corr(y, method='pearson'))
    spearman = float(pd.Series(x).corr(y, method='spearman'))
    # Circular shift p, preserving each series' autocorrelation.
    obs = pearson
    vals = []
    arr = np.asarray(x.fillna(x.mean()))
    target = np.asarray(y)
    for shift in range(1, len(arr)):
        vals.append(pd.Series(np.roll(arr, shift)).corr(pd.Series(target), method='pearson'))
    vals = np.asarray(vals)
    p = float((np.abs(vals) >= abs(obs)).mean())
    monthly_rows.append({'silso_window': label, 'pearson_r': pearson, 'spearman_rho': spearman, 'circular_shift_p_two_sided': p})

monthly_results = pd.DataFrame(monthly_rows)
monthly_results.to_csv(OUT / 'paper_methods_monthly_silso_red.csv', index=False)
monthly_results


## 11. Save summary outputs

This cell writes a compact JSON summary and a plain-language Markdown summary that can be copied into a supplement or project page.


In [ ]:
#@title Save summary files { display-mode: "form" }
summary = {
    'scope': {'start_year': START, 'end_year': END, 'annual_observations': len(df), 'red_episodes_1900_2014': int(df['events'].sum())},
    'primary': {k: primary[k] for k in ['T_w','weighted_distance_years','p_T_rotational','p_distance_rotational','n_rotations']},
    'top_leave_one_year': leave_year.head(12).to_dict(orient='records'),
    'leave_one_cluster': leave_cluster.to_dict(orient='records'),
    'alternative_measures': measure_results.to_dict(orient='records'),
    'threshold_sensitivity': threshold_results.to_dict(orient='records'),
    'monthly_silso': monthly_results.to_dict(orient='records'),
}
(OUT / 'paper_methods_summary.json').write_text(json.dumps(summary, indent=2))

lines = []
lines.append('# Solar Revolutions paper-methods notebook summary')
lines.append('')
lines.append(f"Scope: RED v1.0, {START}-{END}, {int(df['events'].sum())} episode starts, {len(df)} annual observations.")
lines.append('')
lines.append('## Primary phase-alignment result')
lines.append(f"- T_w: {primary['T_w']:.3f}")
lines.append(f"- Weighted mean distance to solar maximum: {primary['weighted_distance_years']:.3f} years")
lines.append(f"- Rotational p for T_w: {primary['p_T_rotational']:.4f}")
lines.append(f"- Rotational p for distance: {primary['p_distance_rotational']:.4f}; if no rotation is more extreme, report conservatively as below the 1/{primary['n_rotations']} resolution floor.")
lines.append('')
lines.append('## Interpretation')
lines.append('- The notebook supports a positive solar-maximum phase alignment for annual RED participation share using public RED v1.0.')
lines.append('- Influence diagnostics show that large historical waves materially affect effect size, although no single removed year reverses the sign.')
lines.append('- Monthly SILSO timing checks are weak/null, so the result should be interpreted as broad cycle-phase alignment rather than month-to-month triggering.')
(OUT / 'paper_methods_summary.md').write_text('\n'.join(lines) + '\n')

print('Wrote outputs to', OUT)
print((OUT / 'paper_methods_summary.md').read_text())


## 12. Recommended citation language

A cautious description of what this notebook establishes:

> Using public RED v1.0 and SILSO solar-cycle extrema, annual revolutionary participation from 1900-2014 is positively concentrated near solar-cycle maxima under a rotational null that preserves the historical clustering of revolutions. The result is strongest for participation-weighted measures, weaker or absent for count-only/log-transformed measures, and materially influenced by several large historical mobilization waves. Monthly SILSO tests do not support a simple short-term triggering interpretation.
